# Colley weighted ranking

Description: Construction of Colley rankings of data with weighted games

### Set parameters

gameFilename - game data file, presumed to be in the format from 
the Massey rating data server, which can be found at 
http://www.masseyratings.com/. 

teamFilename - team data file

k - number of teams to print in the final ranking - set to 0 to get all teams

In [15]:
gameFilename = 'NCAA_2008_Games.txt'
teamFilename = 'NCAA_2008_Teams.txt'
k = 10

In [16]:
# Set weights for home, away and neutral wins
weightHomeWin = 1
weightAwayWin = 1
weightNeutralWin = 1
segmentWeighting = [1/2,2]

# Will you use weighting? 
useWeighting = False 

### Load the team names into an array

In [17]:
import pandas as pd

teamNames = pd.read_csv(teamFilename, header = None)
numTeams = len(teamNames)

### Load the games

In [18]:
# columns of games are:
#	column 0 = days since 1/1/0000
#	column 1 = date in YYYYMMDD format
#	column 2 = team1 index
#	column 3 = team1 homefield (1 = home, -1 = away, 0 = neutral)
#	column 4 = team1 score
#	column 5 = team2 index
#	column 6 = team2 homefield (1 = home, -1 = away, 0 = neutral)
#	column 7 = team2 score
games = pd.read_csv(gameFilename, header = None)
numGames = len(games)

### Create the Colley linear system

In [19]:
import numpy as np
from math import ceil 

colleyMatrix = 2*np.diag(np.ones(numTeams))
b = np.ones(numTeams)

dayBeforeSeason = games.loc[0,0] - 1
lastDayOfSeason = games.loc[len(games)-1,0]

for i in range(numGames):
    team1ID = games.loc[i, 2] - 1 # subtracting 1 since python indexes at 0
    team1Score = games.loc[i, 4]
    team1Loc = games.loc[i, 3];

    team2ID = games.loc[i, 5] - 1 # subtracting 1 since python indexes at 0
    team2Score = games.loc[i, 7]
    team2Loc = games.loc[i, 6];
    
    currentDay = games.loc[i,0]

    # Find the weight for this game using time and home/away    
    if useWeighting:
        numberSegments = len(segmentWeighting)
        weightIndex = ceil(numberSegments*((currentDay-dayBeforeSeason)/(lastDayOfSeason-dayBeforeSeason))) - 1
        timeWeight = segmentWeighting[weightIndex]
    else:
        timeWeight = 1

    if team1Score > team2Score:  # Team 1 won        
        if (team1Loc == 1):      # Home win
            gameWeight = weightHomeWin*timeWeight
        elif (team1Loc == -1):   # Away win
            gameWeight = weightAwayWin*timeWeight
        else:                    # Neutral court win
            gameWeight = weightNeutralWin*timeWeight
    else:                        # Team 2 won
        if (team2Loc == 1):      # Home win
            gameWeight = weightHomeWin*timeWeight
        elif (team2Loc == -1):   # Away win
            gameWeight = weightAwayWin*timeWeight
        else:                    # Neutral court win
            gameWeight = weightNeutralWin*timeWeight
                
    # Update the Colley matrix and RHS
    colleyMatrix[team1ID, team2ID] -= gameWeight
    colleyMatrix[team2ID, team1ID] -= gameWeight

    colleyMatrix[team1ID, team1ID] += gameWeight
    colleyMatrix[team2ID, team2ID] += gameWeight
    
    if team1Score > team2Score:
        b[team1ID] += 1/2*gameWeight
        b[team2ID] -= 1/2*gameWeight
    elif team1Score < team2Score:
        b[team1ID] -= 1/2*gameWeight
        b[team2ID] += 1/2*gameWeight
    else:  # it is a tie and make 1/2 a win and 1/2 a loss for both teams
        b[team1ID] += 0; # this equates to adding nothing
        b[team2ID] += 0; # clearly this code could be deleted

### Calculate linear system

In [20]:
r = np.linalg.solve(colleyMatrix,b)
iSort = np.argsort(-r)

### Print the ranking of the teams

In [21]:
print('\n\n************** COLLEY Rating Method **************\n')
print('===========================')
print('Rank   Rating    Team   ')
print('===========================')
if k==0:
    numberTeamToPrint = numTeams
else:
    numberTeamToPrint = k

for i in range(numberTeamToPrint):
    print(f'{i+1:4d}   {r[iSort[i]]:.5f}  {teamNames.loc[iSort[i],1]}')

print('')   # extra carriage return



************** COLLEY Rating Method **************

Rank   Rating    Team   
   1   1.08453   North_Carolina
   2   1.04626   Memphis
   3   1.03769   UCLA
   4   1.03389   Tennessee
   5   1.01355   Kansas
   6   0.99244   Duke
   7   0.97968   Wisconsin
   8   0.97184   Georgetown
   9   0.97049   Texas
  10   0.94242   Butler



### Calculate predictability of method

In [22]:
numberCorrectPredictions = 0
for i in range(numGames):
    team1ID = games.loc[i, 2] - 1 
    team1Score = games.loc[i, 4]
    team2ID = games.loc[i, 5] - 1 
    team2Score = games.loc[i, 7]
    
    if team1Score > team2Score and r[team1ID] > r[team2ID]:
        numberCorrectPredictions += 1
    elif team2Score > team1Score and r[team2ID] > r[team1ID]:
        numberCorrectPredictions += 1
    elif team1Score == team2Score and r[team1ID] == r[team2ID]:
        numberCorrectPredictions += 1

print(f'Predictability: {numberCorrectPredictions/numGames*100:.2f}%') 


Predictability: 76.12%


In [23]:
# 2008 NCAA bracket prediction using Colley ratings (uniform weighting)
# build quick lookup: team name -> rating
ratings = {str(teamNames.loc[i,1]).strip(): float(r[i]) for i in range(numTeams)}
# mapping tournament names to the naming format in NCAA_2008_Teams.txt
alias = {
    "North Carolina": "North_Carolina",
    "Mount St. Mary's": "Mt_St_Mary's",
    "Arkansas": "Arkansas",
    "Indiana": "Indiana",
    "Notre Dame": "Notre_Dame",
    "George Mason": "George_Mason",
    "Washington State": "Washington_St",
    "Winthrop": "Winthrop",
    "Oklahoma": "Oklahoma",
    "Saint Joseph's": "St_Joseph's_PA",
    "Louisville": "Louisville",
    "Boise State": "Boise_St",
    "Butler": "Butler",
    "South Alabama": "South_Alabama",
    "Tennessee": "Tennessee",
    "American": "American_Univ",


    "Kansas": "Kansas",
    "Portland State": "Portland_St",
    "UNLV": "UNLV",
    "Kent State": "Kent",
    "Clemson": "Clemson",
    "Villanova": "Villanova",
    "Vanderbilt": "Vanderbilt",
    "Siena": "Siena",
    "USC": "USC",
    "Kansas State": "Kansas_St",
    "Wisconsin": "Wisconsin",
    "Cal State Fullerton": "CS_Fullerton",
    "Gonzaga": "Gonzaga",
    "Davidson": "Davidson",
    "Georgetown": "Georgetown",
    "UMBC": "MD_Baltimore_Co",


    "Memphis": "Memphis",
    "Texas-Arlington": "UT_Arlington",
    "Mississippi State": "Mississippi_St",
    "Oregon": "Oregon",
    "Michigan State": "Michigan_St",
    "Temple": "Temple",
    "Pittsburgh": "Pittsburgh",
    "Oral Roberts": "Oral_Roberts",
    "Marquette": "Marquette",
    "Kentucky": "Kentucky",
    "Stanford": "Stanford",
    "Cornell": "Cornell",
    "Miami (FL)": "Miami_FL",
    "Saint Mary's (CA)": "St_Mary's_CA",
    "Texas": "Texas",
    "Austin Peay": "Austin_Peay",


    "UCLA": "UCLA",
    "Mississippi Valley State": "MS_Valley_St",
    "BYU": "BYU",
    "Texas A&M": "Texas_A&M",
    "Drake": "Drake",
    "Western Kentucky": "W_Kentucky",
    "Connecticut": "Connecticut",
    "San Diego": "San_Diego",
    "Purdue": "Purdue",
    "Baylor": "Baylor",
    "Xavier": "Xavier",
    "Georgia": "Georgia",
    "West Virginia": "West_Virginia",
    "Arizona": "Arizona",
    "Duke": "Duke",
    "Belmont": "Belmont"
}


def pick(t1, t2):
    a = alias[t1]
    b = alias[t2]
    r1 = ratings[a]
    r2 = ratings[b]
    return (t1, r1, t2, r2, t1 if r1 >= r2 else t2)


regions = {
    "East": [
        ("North Carolina", "Mount St. Mary's"),
        ("Indiana", "Arkansas"),
        ("Notre Dame", "George Mason"),
        ("Washington State", "Winthrop"),
        ("Oklahoma", "Saint Joseph's"),
        ("Louisville", "Boise State"),
        ("Butler", "South Alabama"),
        ("Tennessee", "American")
    ],
    "Midwest": [
        ("Kansas", "Portland State"),
        ("UNLV", "Kent State"),
        ("Clemson", "Villanova"),
        ("Vanderbilt", "Siena"),
        ("USC", "Kansas State"),
        ("Wisconsin", "Cal State Fullerton"),
        ("Gonzaga", "Davidson"),
        ("Georgetown", "UMBC")
    ],
    "South": [
        ("Memphis", "Texas-Arlington"),
        ("Mississippi State", "Oregon"),
        ("Michigan State", "Temple"),
        ("Pittsburgh", "Oral Roberts"),
        ("Marquette", "Kentucky"),
        ("Stanford", "Cornell"),
        ("Miami (FL)", "Saint Mary's (CA)"),
        ("Texas", "Austin Peay")
    ],
    "West": [
        ("UCLA", "Mississippi Valley State"),
        ("BYU", "Texas A&M"),
        ("Drake", "Western Kentucky"),
        ("Connecticut", "San Diego"),
        ("Purdue", "Baylor"),
        ("Xavier", "Georgia"),
        ("West Virginia", "Arizona"),
        ("Duke", "Belmont")
    ]
}

def advance(round_teams):
    next_round = []
    detailed = []
    for i in range(0, len(round_teams), 2):
        g = pick(round_teams[i], round_teams[i+1])
        detailed.append(g)
        next_round.append(g[4])
    return next_round, detailed

region_champs = {}
print("Colley Bracket Predictions (2008)")
for region, games in regions.items():
    print(f"\n{region} Region")
    r64 = []
    for g in games:
        p = pick(*g)
        r64.append(p[4])
        print(f"R64: {p[0]} ({p[1]:.5f}) vs {p[2]} ({p[3]:.5f}) -> {p[4]}")

    r32, d32 = advance(r64)
    for p in d32:
        print(f"R32: {p[0]} ({p[1]:.5f}) vs {p[2]} ({p[3]:.5f}) -> {p[4]}")

    s16, d16 = advance(r32)
    for p in d16:
        print(f"S16: {p[0]} ({p[1]:.5f}) vs {p[2]} ({p[3]:.5f}) -> {p[4]}")

    e8, d8 = advance(s16)
    p = d8[0]
    print(f"E8 : {p[0]} ({p[1]:.5f}) vs {p[2]} ({p[3]:.5f}) -> {p[4]}")
    region_champs[region] = e8[0]

print("\nFinal Four")
ff1 = pick(region_champs["East"], region_champs["Midwest"])
ff2 = pick(region_champs["South"], region_champs["West"])
print(f"SF1: {ff1[0]} ({ff1[1]:.5f}) vs {ff1[2]} ({ff1[3]:.5f}) -> {ff1[4]}")
print(f"SF2: {ff2[0]} ({ff2[1]:.5f}) vs {ff2[2]} ({ff2[3]:.5f}) -> {ff2[4]}")

title = pick(ff1[4], ff2[4])
print(f"Championship: {title[0]} ({title[1]:.5f}) vs {title[2]} ({title[3]:.5f}) -> {title[4]}")

Colley Bracket Predictions (2008)

East Region
R64: North Carolina (1.08453) vs Mount St. Mary's (0.48312) -> North Carolina
R64: Indiana (0.88603) vs Arkansas (0.80066) -> Indiana
R64: Notre Dame (0.86798) vs George Mason (0.69803) -> Notre Dame
R64: Washington State (0.86201) vs Winthrop (0.60180) -> Washington State
R64: Oklahoma (0.80583) vs Saint Joseph's (0.73820) -> Oklahoma
R64: Louisville (0.90090) vs Boise State (0.68417) -> Louisville
R64: Butler (0.94242) vs South Alabama (0.78044) -> Butler
R64: Tennessee (1.03389) vs American (0.59123) -> Tennessee
R32: North Carolina (1.08453) vs Indiana (0.88603) -> North Carolina
R32: Notre Dame (0.86798) vs Washington State (0.86201) -> Notre Dame
R32: Oklahoma (0.80583) vs Louisville (0.90090) -> Louisville
R32: Butler (0.94242) vs Tennessee (1.03389) -> Tennessee
S16: North Carolina (1.08453) vs Notre Dame (0.86798) -> North Carolina
S16: Louisville (0.90090) vs Tennessee (1.03389) -> Tennessee
E8 : North Carolina (1.08453) vs Tenne

## 3) 2006-07 Pre-March-Madness Ranking (Part a)


This section uses `NCAA_2007_Games.txt` and `NCAA_2007_Teams.txt` only.
It ranks all 336 Division I teams using the Colley method with uniform weighting (all game weights = 1), using games played **before** the 2007 NCAA tournament opening round (`20070313`).

In [24]:
import pandas as pd
import numpy as np
gameFilename_2007 = 'NCAA_2007_Games.txt'
teamFilename_2007 = 'NCAA_2007_Teams.txt'

# we keep only games played before the 2007 NCAA tournament opening round.
march_madness_start_2007 = 20070313

teams07 = pd.read_csv(teamFilename_2007, header=None)
games07_all = pd.read_csv(gameFilename_2007, header=None)
games07 = games07_all[games07_all[1] < march_madness_start_2007].reset_index(drop=True)

numTeams07 = len(teams07)
numGames07 = len(games07)

# uniform-weight Colley system
C07 = 2 * np.eye(numTeams07)
b07 = np.ones(numTeams07)

for i in range(numGames07):
    team1 = int(games07.loc[i, 2]) - 1
    score1 = games07.loc[i, 4]
    team2 = int(games07.loc[i, 5]) - 1
    score2 = games07.loc[i, 7]

    gameWeight = 1.0

    C07[team1, team2] -= gameWeight
    C07[team2, team1] -= gameWeight
    C07[team1, team1] += gameWeight
    C07[team2, team2] += gameWeight

    if score1 > score2:
        b07[team1] += 0.5 * gameWeight
        b07[team2] -= 0.5 * gameWeight
    elif score2 > score1:
        b07[team1] -= 0.5 * gameWeight
        b07[team2] += 0.5 * gameWeight

r07 = np.linalg.solve(C07, b07)
iSort07 = np.argsort(-r07)

print('2006-07 Pre-March Madness Colley Ranking (uniform)')
print(f'Teams ranked: {numTeams07}')
print(f'Games used: {numGames07}')
print('Rank   Rating    Team')

top_k = 15
for i in range(top_k):
    name = str(teams07.loc[iSort07[i], 1]).strip()
    print(f'{i+1:4d}   {r07[iSort07[i]]:.5f}  {name}')

2006-07 Pre-March Madness Colley Ranking (uniform)
Teams ranked: 336
Games used: 5043
Rank   Rating    Team
   1   1.04545  Ohio_St
   2   1.01078  UCLA
   3   1.00199  North_Carolina
   4   0.98207  Wisconsin
   5   0.97741  Kansas
   6   0.96283  Florida
   7   0.96162  Pittsburgh
   8   0.95296  S_Illinois
   9   0.95227  Memphis
  10   0.95193  Georgetown
  11   0.92184  UNLV
  12   0.90158  Oregon
  13   0.89802  Texas_A&M
  14   0.89671  Arizona
  15   0.88583  Duke


## 3) 2006-07 Pre-March-Madness Ranking (Part b: Custom Weighting)

Custom weighting used in this section:
- Time weight (linear): increases from `m1 = 0.70` at start of season to `m2 = 1.30` at the March Madness cutoff.
- Location multiplier by winner: away win `1.10`, neutral win `1.00`, home win `0.95`.

Final game weight = `time_weight * location_multiplier_of_winner`.

In [25]:
import numpy as np
# just re-using pre-tournament games from part a
if 'games07' not in globals() or 'teams07' not in globals():
    teams07 = pd.read_csv('NCAA_2007_Teams.txt', header=None)
    games07_all = pd.read_csv('NCAA_2007_Games.txt', header=None)
    march_madness_start_2007 = 20070313
    games07 = games07_all[games07_all[1] < march_madness_start_2007].reset_index(drop=True)

numTeams07 = len(teams07)
numGames07 = len(games07)

# my weighting parameters
m1 = 0.70
m2 = 1.30
weight_home_win = 0.95
weight_away_win = 1.10
weight_neutral_win = 1.00

day_min = int(games07.loc[0, 0])
day_max = int(games07.loc[numGames07 - 1, 0])

C07_custom = 2 * np.eye(numTeams07)
b07_custom = np.ones(numTeams07)

for i in range(numGames07):
    team1 = int(games07.loc[i, 2]) - 1
    loc1 = int(games07.loc[i, 3])
    score1 = games07.loc[i, 4]
    team2 = int(games07.loc[i, 5]) - 1
    loc2 = int(games07.loc[i, 6])
    score2 = games07.loc[i, 7]
    day = int(games07.loc[i, 0])

    # linear-time scaling from m1 to m2 across the season
    if day_max == day_min:
        time_weight = m2
    else:
        alpha = (day - day_min) / (day_max - day_min)
        time_weight = m1 + (m2 - m1) * alpha

    # winner-dependent location multiplier
    if score1 > score2:
        if loc1 == 1:
            location_weight = weight_home_win
        elif loc1 == -1:
            location_weight = weight_away_win
        else:
            location_weight = weight_neutral_win
    elif score2 > score1:
        if loc2 == 1:
            location_weight = weight_home_win
        elif loc2 == -1:
            location_weight = weight_away_win
        else:
            location_weight = weight_neutral_win
    else:
        location_weight = weight_neutral_win

    gameWeight = time_weight * location_weight

    C07_custom[team1, team2] -= gameWeight
    C07_custom[team2, team1] -= gameWeight
    C07_custom[team1, team1] += gameWeight
    C07_custom[team2, team2] += gameWeight

    if score1 > score2:
        b07_custom[team1] += 0.5 * gameWeight
        b07_custom[team2] -= 0.5 * gameWeight
    elif score2 > score1:
        b07_custom[team1] -= 0.5 * gameWeight
        b07_custom[team2] += 0.5 * gameWeight

r07_custom = np.linalg.solve(C07_custom, b07_custom)
iSort07_custom = np.argsort(-r07_custom)

print('2006-07 Pre-March Madness Colley Ranking (CUSTOM WEIGHTING)')
print(f'Teams ranked: {numTeams07}')
print(f'Games used: {numGames07}')
print(f'm1={m1:.2f}, m2={m2:.2f}, home={weight_home_win:.2f}, away={weight_away_win:.2f}, neutral={weight_neutral_win:.2f}')
print('Rank   Rating    Team')

top_k_custom = 15
for i in range(top_k_custom):
    name = str(teams07.loc[iSort07_custom[i], 1]).strip()
    print(f'{i+1:4d}   {r07_custom[iSort07_custom[i]]:.5f}  {name}')

2006-07 Pre-March Madness Colley Ranking (CUSTOM WEIGHTING)
Teams ranked: 336
Games used: 5043
m1=0.70, m2=1.30, home=0.95, away=1.10, neutral=1.00
Rank   Rating    Team
   1   1.06620  Ohio_St
   2   0.99871  North_Carolina
   3   0.99324  Kansas
   4   0.98591  UCLA
   5   0.98412  Florida
   6   0.98159  Wisconsin
   7   0.97570  Georgetown
   8   0.96674  S_Illinois
   9   0.95824  Memphis
  10   0.95079  Pittsburgh
  11   0.92919  UNLV
  12   0.89845  Texas_A&M
  13   0.89283  Oregon
  14   0.88372  Nevada
  15   0.87947  Washington_St
